In [ ]:
# ======================================================================
# AAI-520 Team 8 Team Project — (Brandon, Christina and Isaac) Task Status
# ======================================================================
#
# ======================================================================
# Isaac — Market & Planning
# ======================================================================
#
# 1. Implement Router
#
# 2. Implement Planner Agent
#
# 3. Build Yahoo Finance market data tools
#
# 4. Calculate market metrics
#
# 5. Implement Market Agent
#
# 6. Additional Person 1 task(s) hidden in screenshot
#
# ======================================================================
# Brandon — Financial & Evaluation
# ======================================================================
#
# 1. Build financial statements tool
#
# 2. Implement Evaluator Agent
#
# 3. Implement Self-Reflection
#
# 4. Implement Optimizer
#
# 5. Define evaluation criteria / rubric
#
# 6. Implement Financial Agent
#
# ======================================================================
# Brandon, Christina and Isaac
# ======================================================================
#
# 1. End-to-end demo + learning-across-runs demo
#
# 2. Configure dependencies & environment
#
# 3. Full system testing
#    (component, routing, evaluator, memory)
#
# 4. Define shared LangGraph state schema
#
# 5. Finalize project scope & MVP definition
#
# 6. Additional Whole Group task(s) hidden in screenshot
#
# ======================================================================
# Christina — News & Memory
# ======================================================================
#
# 1. Integrate memory with planning
#
# 2. Implement Synthesis Agent
#
# 3. Implement persistent memory
#
# 4. Build news retrieval tool
#
# 5. Build prompt chaining news workflow
#    (ingest to summarize)
#
# 6. Additional Person 3 task(s) hidden in screenshot
#
# ======================================================================
# End of Task Status
# ======================================================================

In [7]:
#
print("\033[1m Team Members: Brandon, Christina and Isaac\033[0m")
print()
print("\033[1mTask: Configure dependencies & environment\033[0m")
print()
#

%pip install "pandas<3" ollama yfinance


 Team Members: Brandon, Christina and Isaac

Task: Configure dependencies & environment




In [8]:
#
print("\033[1m Team Member: Brandon\033[0m")
print()
print("\033[1mTask: Ollama \033[0m")
print()
#

import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "Explain what a P/E ratio means in one sentence."
        }
    ]
)

print(response["message"]["content"])


 Team Member: Brandon

Task: Ollama 

The P/E ratio, or price-to-earnings ratio, is a financial metric that compares a company's current stock price to its earnings per share, providing investors with a general idea of whether the stock is overvalued, undervalued, or fairly priced relative to its earnings.


In [9]:
#
print("\033[1m Team Member: Brandon\033[0m")
print()
print("\033[1mTask: Ollama \033[0m")
print()
#

import ollama

models = ollama.list()

for model in models["models"]:
    print(model["model"])


 Team Member: Brandon

Task: Ollama 

llama3.2:latest


In [10]:
#
print("\033[1m Team Member: Brandon\033[0m")
print()
print("\033[1mTask: Ollama \033[0m")
print()
#

import ollama

print(ollama.list())


 Team Member: Brandon

Task: Ollama 

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 9, 16, 21, 34, 32, 419930, tzinfo=TzInfo(-14400)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]


In [11]:
#
print("\033[1m Team Member: Brandon\033[0m")
print()
print("\033[1mTask: Ollama \033[0m")
print()
#

import json
from pathlib import Path
from datetime import datetime

import ollama
import yfinance as yf
import pandas as pd

MODEL = "llama3.2:latest"
MEMORY_FILE = Path("market_agent_memory.json")

print("Imports loaded")
print(f"Local model: {MODEL}")


 Team Member: Brandon

Task: Ollama 

Imports loaded
Local model: llama3.2:latest


In [12]:
#
print("\033[1m Team Member: Brandon\033[0m")
print()
print("\033[1mTask: Ollama \033[0m")
print()
#

def ask_llm(prompt):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    
    return response["message"]["content"]


test = ask_llm(
    "You are a financial research assistant. "
    "Give me a 3-step research plan for AAPL."
)

print(test)


 Team Member: Brandon

Task: Ollama 

As a financial research assistant, I'd be happy to help you with a 3-step research plan for AAPL (Apple Inc.). Here's a plan to get you started:

**Step 1: Review Historical Performance and Financial Metrics**

* Gather historical stock price data for AAPL, including daily, weekly, and monthly charts.
* Review Apple's financial statements, including the income statement, balance sheet, and cash flow statement, for the past 5-7 years.
* Calculate key financial metrics such as:
	+ Revenue growth rate
	+ Gross margin percentage
	+ Operating income margin
	+ Return on equity (ROE)
	+ Debt-to-equity ratio
* Use online resources such as Yahoo Finance, Google Finance, or Quandl to access these data points.

**Step 2: Analyze Industry Trends and Competitors**

* Research the technology and smartphone industries, focusing on trends, challenges, and opportunities.
* Analyze the competitive landscape, including:
	+ Key competitors: Samsung, Huawei, Google, Am

In [13]:
#
print("\033[1m Team Member: Isaac\033[0m")
print()
print("\033[1mTask: Yahoo Finance + market metrics + financial statements \033[0m")
print()
#

def get_price_data(symbol, period="1y"):
    ticker = yf.Ticker(symbol)
    df = ticker.history(period=period)

    if df.empty:
        return {"error": f"No price data found for {symbol}"}

    close = df["Close"]

    start_price = float(close.iloc[0])
    latest_price = float(close.iloc[-1])

    total_return = (
        latest_price / start_price - 1
    ) * 100

    daily_returns = close.pct_change().dropna()

    volatility = (
        daily_returns.std() * (252 ** 0.5)
    ) * 100

    running_max = close.cummax()

    drawdown = (
        close / running_max - 1
    ) * 100

    return {
        "start_price": round(start_price, 2),
        "latest_price": round(latest_price, 2),
        "1y_return_percent": round(total_return, 2),
        "annualized_volatility_percent": round(
            float(volatility), 2
        ),
        "maximum_drawdown_percent": round(
            float(drawdown.min()), 2
        ),
        "observations": len(df),
    }


def get_company_info(symbol):
    ticker = yf.Ticker(symbol)

    info = ticker.info

    fields = [
        "longName",
        "sector",
        "industry",
        "country",
        "marketCap",
        "enterpriseValue",
        "currentPrice",
        "trailingPE",
        "forwardPE",
        "priceToSalesTrailing12Months",
        "profitMargins",
        "operatingMargins",
        "returnOnEquity",
        "beta",
        "dividendYield",
    ]

    return {
        field: info.get(field)
        for field in fields
    }


def get_financials(symbol):
    ticker = yf.Ticker(symbol)

    income = ticker.income_stmt

    if income.empty:
        return {"error": "No financial statements found"}

    wanted = [
        "Total Revenue",
        "Operating Income",
        "Net Income",
        "EBITDA",
        "Diluted EPS",
    ]

    available = [
        row for row in wanted
        if row in income.index
    ]

    result = {}

    for row in available:
        values = income.loc[row].iloc[:4]

        result[row] = {
            str(date): (
                None if pd.isna(value)
                else float(value)
            )
            for date, value in values.items()
        }

    return result


def get_cash_flow(symbol):
    ticker = yf.Ticker(symbol)

    cashflow = ticker.cashflow

    if cashflow.empty:
        return {"error": "No cash-flow data found"}

    wanted = [
        "Operating Cash Flow",
        "Free Cash Flow",
        "Capital Expenditure",
    ]

    available = [
        row for row in wanted
        if row in cashflow.index
    ]

    result = {}

    for row in available:
        values = cashflow.loc[row].iloc[:4]

        result[row] = {
            str(date): (
                None if pd.isna(value)
                else float(value)
            )
            for date, value in values.items()
        }

    return result
    

 Team Member: Isaac

Task: Yahoo Finance + market metrics + financial statements 



In [14]:
#
print("\033[1m Team Member: Isaac\033[0m")
print()
print("\033[1mTask: Yahoo Finance + market metrics + financial statements \033[0m")
print()
#

TOOLS = {
    "price_data": get_price_data,
    "company_info": get_company_info,
    "financials": get_financials,
    "cash_flow": get_cash_flow,
}

print("Available tools:")

for name in TOOLS:
    print(" -", name)


 Team Member: Isaac

Task: Yahoo Finance + market metrics + financial statements 

Available tools:
 - price_data
 - company_info
 - financials
 - cash_flow


In [15]:
#
print("\033[1m Team Member: Christina\033[0m")
print()
print("\033[1mTask: Memory integration + persistent memory \033[0m")
print()
#


def load_memory():
    if not MEMORY_FILE.exists():
        return []

    try:
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def save_memory(symbol, summary, weaknesses):
    memories = load_memory()

    memories.append({
        "date": datetime.now().isoformat(),
        "symbol": symbol,
        "summary": summary,
        "weaknesses": weaknesses,
    })

    # Keep the most recent 20 memories
    memories = memories[-20:]

    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memories, f, indent=2)


def get_memory(symbol):
    memories = load_memory()

    return [
        m for m in memories
        if m.get("symbol") == symbol
    ]


 Team Member: Christina

Task: Memory integration + persistent memory 



In [16]:
#
print("\033[1m Team Member: Christina\033[0m")
print()
print("\033[1mTask: Install matplotlib \033[0m")
print()
#

%pip install matplotlib


 Team Member: Christina

Task: Install matplotlib 




In [17]:
#
print("\033[1m Team Members: Brandon, Christina and Isaac\033[0m")
print()
print("\033[1mBuild an autonomous Investment Research Agent that:  \033[0m")
print("\033[1m 1. Plans its research steps for a given stock symbol.\033[0m")
print("\033[1m 2. Uses tools dynamically (APIs, datasets, retrieval). \033[0m")
print("\033[1m 3. Self-reflects to assess the quality of its output.  \033[0m")
print("\033[1m 4. Learns across runs (e.g., keeps brief memories or notes to improve future analyses).  \033[0m")
print()
#


import json
import re
from numbers import Number

import yfinance as yf


class MarketResearchAgent:
    def __init__(self):
        self.model = MODEL

    # =========================================================
    # LLM
    # =========================================================

    def ask(self, prompt):
        return ask_llm(prompt)

    # =========================================================
    # JSON PARSER
    # =========================================================

    def parse_json_response(self, response, fallback=None):
        """
        Safely parse JSON returned by an LLM.

        Handles:
        - normal JSON
        - Markdown code fences
        - explanatory text before/after JSON
        """

        if not response or not isinstance(response, str):
            return fallback

        text = response.strip()

        # Attempt 1: direct JSON
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Remove Markdown code fences
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        ).strip()

        # Attempt 2: JSON after removing fences
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Attempt 3: extract JSON object
        start = text.find("{")
        end = text.rfind("}")

        if start != -1 and end > start:
            candidate = text[start:end + 1]

            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass

        return fallback

    # =========================================================
    # PLAN
    # =========================================================

    def plan(self, symbol):
        previous_memory = get_memory(symbol)

        if previous_memory is None:
            previous_memory = []

        if not isinstance(previous_memory, list):
            previous_memory = [previous_memory]

        recent_memory = previous_memory[-3:]
        available_tools = list(TOOLS.keys())

        prompt = f"""
You are the planning component of a financial research agent.

Stock symbol:
{symbol}

Previous research memory:
{json.dumps(recent_memory, indent=2, default=str)}

Available tools:
{json.dumps(available_tools, indent=2)}

Create a research plan.

Rules:

1. Only select tools from the available tool list.
2. Do not invent tools.
3. Prefer tools that directly answer the research objectives.
4. Use previous memory to avoid repeating unresolved work.
5. Do not make investment recommendations.
6. Do not assume data exists before the tools are executed.

Return ONLY a valid JSON object.

Required schema:

{{
    "objectives": [],
    "tools": [],
    "questions": []
}}
"""

        response = self.ask(prompt)

        fallback = {
            "objectives": [
                "Analyze price performance",
                "Analyze company information",
                "Analyze valuation",
                "Analyze financial performance",
                "Analyze cash generation",
            ],
            "tools": [
                tool
                for tool in [
                    "price_data",
                    "company_info",
                    "financials",
                    "cash_flow",
                ]
                if tool in TOOLS
            ],
            "questions": [
                "How has the stock performed?",
                "What is the company's business and operating profile?",
                "What valuation metrics are available?",
                "How are revenue and earnings performing?",
                "What does the available cash-flow data show?",
            ],
        }

        plan = self.parse_json_response(response, fallback)

        if not isinstance(plan, dict):
            plan = fallback

        objectives = plan.get("objectives", [])
        tools = plan.get("tools", [])
        questions = plan.get("questions", [])

        if not isinstance(objectives, list):
            objectives = []

        if not isinstance(tools, list):
            tools = []

        if not isinstance(questions, list):
            questions = []

        # Only allow actual tools
        tools = [
            tool
            for tool in tools
            if isinstance(tool, str) and tool in TOOLS
        ]

        if not tools:
            tools = fallback["tools"]

        return {
            "objectives": objectives,
            "tools": tools,
            "questions": questions,
        }

    # =========================================================
    # TOOL EXECUTION
    # =========================================================

    def execute_tools(self, symbol, plan):
        observations = {}

        for tool_name in plan.get("tools", []):
            print(f"🔧 Running tool: {tool_name}")

            if tool_name not in TOOLS:
                observations[tool_name] = {
                    "error": "Tool is not available."
                }
                continue

            try:
                tool = TOOLS[tool_name]
                result = tool(symbol)

                if result is None:
                    result = {
                        "error": "Tool returned no data."
                    }

                observations[tool_name] = result

            except Exception as exc:
                observations[tool_name] = {
                    "error": f"{type(exc).__name__}: {exc}"
                }

        return observations

    # =========================================================
    # DATA VALIDATION
    # =========================================================

    def validate_observations(self, symbol, observations):
        issues = []

        def find_value(obj, possible_keys):
            if isinstance(obj, dict):
                for key in possible_keys:
                    if key in obj:
                        return obj[key]

                for value in obj.values():
                    result = find_value(value, possible_keys)

                    if result is not None:
                        return result

            elif isinstance(obj, list):
                for item in obj:
                    result = find_value(item, possible_keys)

                    if result is not None:
                        return result

            return None

        # -----------------------------------------------------
        # Tool-level validation
        # -----------------------------------------------------

        for tool_name, data in observations.items():
            if data is None:
                issues.append({
                    "type": "missing",
                    "tool": tool_name,
                    "message": "Tool returned None.",
                })
                continue

            if isinstance(data, dict) and "error" in data:
                issues.append({
                    "type": "tool_error",
                    "tool": tool_name,
                    "message": str(data["error"]),
                })

        company = observations.get("company_info", {})
        financials = observations.get("financials", {})
        cash_flow = observations.get("cash_flow", {})
        price = observations.get("price_data", {})

        # -----------------------------------------------------
        # Locate important values
        # -----------------------------------------------------

        market_cap = find_value(
            company,
            [
                "market_cap",
                "marketCap",
                "marketCapitalization",
            ],
        )

        revenue = find_value(
            financials,
            [
                "total_revenue",
                "totalRevenue",
                "revenue",
                "Total Revenue",
            ],
        )

        operating_cash_flow = find_value(
            cash_flow,
            [
                "operating_cash_flow",
                "operatingCashFlow",
                "operating_cashflow",
                "Operating Cash Flow",
            ],
        )

        free_cash_flow = find_value(
            cash_flow,
            [
                "free_cash_flow",
                "freeCashFlow",
                "Free Cash Flow",
            ],
        )

        latest_price = find_value(
            price,
            [
                "latest_price",
                "current_price",
                "currentPrice",
                "regularMarketPrice",
                "price",
                "Current Price",
            ],
        )

        one_year_return = find_value(
            price,
            [
                "one_year_return",
                "one_year_return_pct",
                "1y_return",
                "return_1y",
            ],
        )

        # -----------------------------------------------------
        # Cash-flow consistency
        # -----------------------------------------------------

        if operating_cash_flow is not None and free_cash_flow is None:
            issues.append({
                "type": "missing",
                "field": "free_cash_flow",
                "message": (
                    "Operating cash flow exists, but free cash flow "
                    "is not available."
                ),
            })

        # -----------------------------------------------------
        # Price validation
        # -----------------------------------------------------

        if latest_price is None:
            issues.append({
                "type": "missing",
                "field": "latest_price",
                "message": "Latest stock price was not supplied.",
            })

        # -----------------------------------------------------
        # Return validation
        # -----------------------------------------------------

        if one_year_return is None:
            issues.append({
                "type": "missing",
                "field": "one_year_return",
                "message": "One-year return was not supplied.",
            })

        # -----------------------------------------------------
        # Market-cap sanity check
        # -----------------------------------------------------

        if (
            isinstance(market_cap, Number)
            and isinstance(revenue, Number)
            and revenue > 0
        ):
            ratio = market_cap / revenue

            if ratio > 1000:
                issues.append({
                    "type": "suspicious",
                    "field": "market_cap",
                    "message": (
                        "Market capitalization is more than 1000x "
                        "reported revenue. Verify units and scaling "
                        "before using this value."
                    ),
                    "market_cap": market_cap,
                    "revenue": revenue,
                    "ratio": ratio,
                })

        # -----------------------------------------------------
        # Percentage sanity checks
        # -----------------------------------------------------

        def check_percent_field(data, keys):
            value = find_value(data, keys)

            if isinstance(value, Number) and abs(value) > 1000:
                issues.append({
                    "type": "suspicious",
                    "field": keys[0],
                    "message": (
                        "Percentage-like value is unusually large. "
                        "Verify units."
                    ),
                    "value": value,
                })

        check_percent_field(
            price,
            [
                "one_year_return",
                "one_year_return_pct",
            ],
        )

        check_percent_field(
            company,
            [
                "profit_margin",
                "operating_margin",
                "dividend_yield",
                "profitMargins",
                "operatingMargins",
                "dividendYield",
            ],
        )

        return issues

    # =========================================================
    # REFLECTION
    # =========================================================

    def reflect(
        self,
        symbol,
        plan,
        observations,
        validation_issues,
    ):
        prompt = f"""
You are the data-quality auditor of a financial research agent.

Stock:
{symbol}

Research plan:
{json.dumps(plan, indent=2, default=str)}

Collected data:
{json.dumps(observations, indent=2, default=str)}

Automated validation findings:
{json.dumps(validation_issues, indent=2, default=str)}

Your task is to inspect the supplied data carefully.

Rules:

1. Inspect the actual supplied fields.
2. Never claim a value is missing if it exists anywhere in the data.
3. Distinguish missing information from suspicious information.
4. Do not invent values.
5. Do not infer future values.
6. Check units such as dollars, thousands, millions, billions,
   percentages, and ratios.
7. Check whether financial figures appear internally consistent.
8. Treat suspicious values as requiring verification, not automatically
   as incorrect.
9. Do not ask questions unrelated to the research.
10. Do not provide investment advice.
11. Do not evaluate whether the stock should be bought or sold.

Return ONLY valid JSON:

{{
    "strengths": [],
    "weaknesses": [],
    "missing_information": [],
    "suspicious_values": [],
    "follow_up_questions": []
}}
"""

        response = self.ask(prompt)

        fallback = {
            "strengths": [],
            "weaknesses": [],
            "missing_information": [],
            "suspicious_values": [],
            "follow_up_questions": [],
        }

        reflection = self.parse_json_response(response, fallback)

        if not isinstance(reflection, dict):
            return fallback

        fields = [
            "strengths",
            "weaknesses",
            "missing_information",
            "suspicious_values",
            "follow_up_questions",
        ]

        for field in fields:
            if not isinstance(reflection.get(field), list):
                reflection[field] = []

        # Add deterministic validation findings
        for issue in validation_issues:
            issue_type = issue.get("type")
            message = issue.get("message")

            if issue_type == "missing":
                if message not in reflection["missing_information"]:
                    reflection["missing_information"].append(message)

            elif issue_type == "suspicious":
                if message not in reflection["suspicious_values"]:
                    reflection["suspicious_values"].append(message)

            elif issue_type == "tool_error":
                if message not in reflection["weaknesses"]:
                    reflection["weaknesses"].append(message)

        return reflection

    # =========================================================
    # REPORT
    # =========================================================

    def report(
        self,
        symbol,
        plan,
        observations,
        reflection,
        validation_issues,
    ):
        prompt = f"""
You are a neutral financial research reporting system.

Create a factual research report for {symbol}.

Research plan:
{json.dumps(plan, indent=2, default=str)}

Collected tool observations:
{json.dumps(observations, indent=2, default=str)}

Automated validation:
{json.dumps(validation_issues, indent=2, default=str)}

Quality review:
{json.dumps(reflection, indent=2, default=str)}

Structure:

# {symbol} Market Research

## 1. Company Overview

## 2. Price Performance

## 3. Valuation

## 4. Financial Performance

## 5. Cash Flow

## 6. Risks and Uncertainties

## 7. Data Quality

## 8. Further Research

STRICT RULES:

- Use ONLY information supplied in the observations.
- Do not use outside knowledge.
- Do not invent missing values.
- Do not estimate missing values.
- Do not silently correct suspicious values.
- Preserve the units supplied by the tools.
- Every numerical claim must be traceable to supplied data.
- If a number has been flagged as suspicious, explicitly identify it
  as requiring verification.
- Do not claim information is missing when it exists in the observations.
- Distinguish factual observations from interpretation.
- Do not calculate correlations.
- Do not make predictions unless the supplied data explicitly contains
  a prediction.
- Do not provide personalized investment advice.
- Do not say buy, sell, or hold.
- Do not give an overall investment rating.

Important:

If the observations contain operating cash flow or free cash flow,
the Cash Flow section MUST discuss those values.

If the latest stock price is missing, say that it is unavailable.

If a valuation number looks suspicious, report it as supplied and
clearly flag the unit/scaling issue rather than replacing it.

Return the report as Markdown.
"""

        return self.ask(prompt)

    # =========================================================
    # REPORT VALIDATION
    # =========================================================

    def validate_report(
        self,
        symbol,
        report,
        observations,
    ):
        issues = []

        if not report or not isinstance(report, str):
            return [
                "Report generation returned an empty or invalid response."
            ]

        report_lower = report.lower()

        cash_flow = observations.get("cash_flow", {})

        def contains_key_recursive(obj, keys):
            if isinstance(obj, dict):
                for key, value in obj.items():
                    if key in keys:
                        return True

                    if contains_key_recursive(value, keys):
                        return True

            elif isinstance(obj, list):
                for item in obj:
                    if contains_key_recursive(item, keys):
                        return True

            return False

        has_cash_data = contains_key_recursive(
            cash_flow,
            {
                "operating_cash_flow",
                "operatingCashFlow",
                "Operating Cash Flow",
                "free_cash_flow",
                "freeCashFlow",
                "Free Cash Flow",
            },
        )

        if has_cash_data:
            missing_phrases = [
                "cash generation data is missing",
                "cash flow data is missing",
                "cash generation is missing",
            ]

            for phrase in missing_phrases:
                if phrase in report_lower:
                    issues.append(
                        "Report incorrectly claims cash-flow data is missing."
                    )

        prohibited_phrases = [
            "buy the stock",
            "sell the stock",
            "you should buy",
            "you should sell",
            "strong buy",
            "strong sell",
        ]

        for phrase in prohibited_phrases:
            if phrase in report_lower:
                issues.append(
                    f"Report contains unsupported investment language: {phrase}"
                )

        return issues

    # =========================================================
    # MEMORY
    # =========================================================

    def remember(
        self,
        symbol,
        report,
        reflection,
    ):
        prompt = f"""
Summarize this financial research for future research sessions.

Report:
{report}

Quality review:
{json.dumps(reflection, indent=2, default=str)}

Create 3-5 concise notes.

Focus on:

- important factual findings
- unresolved research questions
- suspicious or questionable data
- data limitations
- useful future research

Rules:

- Do not invent information.
- Do not provide investment advice.
- Do not say buy, sell, or hold.
- Keep each note concise.

Return plain text.
"""

        summary = self.ask(prompt)

        if not summary:
            summary = "No research summary was generated."

        save_memory(
            symbol,
            summary,
            reflection.get("weaknesses", []),
        )

    # =========================================================
    # FORMATTERS
    # =========================================================

    @staticmethod
    def format_billions(value):
        if not isinstance(value, Number):
            return "N/A"

        return f"${value / 1_000_000_000:.3f}B"

    @staticmethod
    def format_trillions(value):
        if not isinstance(value, Number):
            return "N/A"

        return f"${value / 1_000_000_000_000:.3f}T"

    @staticmethod
    def format_number(value, decimals=2):
        if not isinstance(value, Number):
            return "N/A"

        return f"{value:,.{decimals}f}"

    @staticmethod
    def format_percent(value, decimals=2):
        if not isinstance(value, Number):
            return "N/A"

        return f"{value * 100:.{decimals}f}%"

    @staticmethod
    def format_ratio(a, b):
        if (
            not isinstance(a, Number)
            or not isinstance(b, Number)
            or b == 0
        ):
            return "N/A"

        return f"{a / b:.3f}x"

    @staticmethod
    def get_latest_year(data):
        """
        Safely determine the newest year/key from a financial dictionary.
        """

        if not isinstance(data, dict) or not data:
            return None

        keys = list(data.keys())

        year_keys = []

        for key in keys:
            match = re.search(r"(19|20)\d{2}", str(key))

            if match:
                year_keys.append((match.group(0), key))

        if year_keys:
            year_keys.sort(
                key=lambda item: item[0],
                reverse=True,
            )
            return year_keys[0][1]

        return keys[0]

    @staticmethod
    def get_nested(data, key, default=None):
        if not isinstance(data, dict):
            return default

        return data.get(key, default)

    # =========================================================
    # CRISP FINAL REPORT
    # =========================================================

    def print_final_report(self, research_result):
        symbol = research_result.get(
            "symbol",
            "UNKNOWN",
        )

        observations = research_result.get(
            "observations",
            {},
        )

        info = observations.get(
            "company_info",
            {},
        )

        financials = observations.get(
            "financials",
            {},
        )

        cash_flow = observations.get(
            "cash_flow",
            {},
        )

        validation = research_result.get(
            "validation",
            [],
        )

        # -----------------------------------------------------
        # Latest financial year
        # -----------------------------------------------------

        latest_year = self.get_latest_year(financials)

        if latest_year is None:
            latest_year = self.get_latest_year(cash_flow)

        # -----------------------------------------------------
        # Financial values
        # -----------------------------------------------------

        revenue = None
        operating_income = None
        net_income = None
        ebitda = None
        eps = None

        if latest_year:
            revenue = self.get_nested(
                financials.get("Total Revenue", {}),
                latest_year,
            )

            operating_income = self.get_nested(
                financials.get("Operating Income", {}),
                latest_year,
            )

            net_income = self.get_nested(
                financials.get("Net Income", {}),
                latest_year,
            )

            ebitda = self.get_nested(
                financials.get("EBITDA", {}),
                latest_year,
            )

            eps = self.get_nested(
                financials.get("Diluted EPS", {}),
                latest_year,
            )

        # -----------------------------------------------------
        # Cash flow
        # -----------------------------------------------------

        operating_cf = None
        free_cf = None
        capex = None

        if latest_year:
            operating_cf = self.get_nested(
                cash_flow.get("Operating Cash Flow", {}),
                latest_year,
            )

            free_cf = self.get_nested(
                cash_flow.get("Free Cash Flow", {}),
                latest_year,
            )

            capex = self.get_nested(
                cash_flow.get("Capital Expenditure", {}),
                latest_year,
            )

        # -----------------------------------------------------
        # Company fields
        # -----------------------------------------------------

        long_name = info.get(
            "longName",
            symbol,
        )

        sector = info.get(
            "sector",
            "N/A",
        )

        industry = info.get(
            "industry",
            "N/A",
        )

        country = info.get(
            "country",
            "N/A",
        )

        current_price = info.get("currentPrice")
        market_cap = info.get("marketCap")
        enterprise_value = info.get("enterpriseValue")

        trailing_pe = info.get("trailingPE")
        forward_pe = info.get("forwardPE")

        price_to_sales = info.get(
            "priceToSalesTrailing12Months"
        )

        profit_margin = info.get("profitMargins")
        operating_margin = info.get("operatingMargins")
        roe = info.get("returnOnEquity")
        beta = info.get("beta")
        dividend_yield = info.get("dividendYield")

        # -----------------------------------------------------
        # Print
        # -----------------------------------------------------

        print("=" * 70)
        print("FINAL RESEARCH REPORT")
        print("=" * 70)

        print(f"\n{long_name} ({symbol})")
        print("-" * 70)

        print("\n1. COMPANY OVERVIEW")

        print(f"   Sector:              {sector}")
        print(f"   Industry:            {industry}")
        print(f"   Country:             {country}")

        print(
            f"   Current Price:       "
            f"${self.format_number(current_price)}"
        )

        print(
            f"   Market Cap:          "
            f"{self.format_trillions(market_cap)}"
        )

        print(
            f"   Enterprise Value:    "
            f"{self.format_trillions(enterprise_value)}"
        )

        print("\n2. VALUATION")

        print(
            f"   Trailing P/E:        "
            f"{self.format_number(trailing_pe)}"
        )

        print(
            f"   Forward P/E:         "
            f"{self.format_number(forward_pe)}"
        )

        print(
            f"   Price/Sales (TTM):   "
            f"{self.format_number(price_to_sales)}"
        )

        print(
            f"   Profit Margin:       "
            f"{self.format_percent(profit_margin)}"
        )

        print(
            f"   Operating Margin:    "
            f"{self.format_percent(operating_margin)}"
        )

        print(
            f"   Return on Equity:    "
            f"{self.format_percent(roe)}"
        )

        print(
            f"   Beta:                "
            f"{self.format_number(beta, 3)}"
        )

        year_label = (
            str(latest_year)[:4]
            if latest_year
            else "N/A"
        )

        print(
            f"\n3. FINANCIAL PERFORMANCE ({year_label})"
        )

        print(
            f"   Revenue:             "
            f"{self.format_billions(revenue)}"
        )

        print(
            f"   Operating Income:    "
            f"{self.format_billions(operating_income)}"
        )

        print(
            f"   Net Income:          "
            f"{self.format_billions(net_income)}"
        )

        print(
            f"   EBITDA:              "
            f"{self.format_billions(ebitda)}"
        )

        print(
            f"   Diluted EPS:         "
            f"${self.format_number(eps)}"
        )

        print("\n4. EBITDA / NET INCOME ANALYSIS")

        print(
            f"   EBITDA:              "
            f"{self.format_billions(ebitda)}"
        )

        print(
            f"   Net Income:          "
            f"{self.format_billions(net_income)}"
        )

        print(
            f"   EBITDA / Net Income: "
            f"{self.format_ratio(ebitda, net_income)}"
        )

        print(f"\n5. CASH FLOW ({year_label})")

        print(
            f"   Operating Cash Flow: "
            f"{self.format_billions(operating_cf)}"
        )

        print(
            f"   Free Cash Flow:      "
            f"{self.format_billions(free_cf)}"
        )

        print(
            f"   Capital Expenditure: "
            f"{self.format_billions(capex)}"
        )

        print("\n6. DIVIDEND")

        print(
            f"   Dividend Yield:      "
            f"{self.format_percent(dividend_yield)}"
        )

        print("\n7. DATA VALIDATION")

        if validation:
            for item in validation:
                field = item.get(
                    "field",
                    "general",
                )

                message = item.get(
                    "message",
                    str(item),
                )

                print(
                    f"   - {field}: {message}"
                )
        else:
            print("   No validation issues reported.")

        missing = [
            item.get("field", "unknown")
            for item in validation
            if item.get("type") == "missing"
        ]

        print("\n8. MISSING INFORMATION")

        if missing:
            for field in missing:
                print(f"   - {field}")
        else:
            print("   None")

        print("\n" + "=" * 70)
        print("END OF FINAL RESEARCH REPORT")
        print("=" * 70)

    # =========================================================
    # MAIN RUNNER
    # =========================================================

    def run(self, symbol):
        symbol = symbol.upper().strip()

        if not symbol:
            raise ValueError(
                "Stock symbol cannot be empty."
            )

        print("=" * 60)
        print(f"AGENTIC MARKET RESEARCH: {symbol}")
        print("=" * 60)

        # =====================================================
        # 1. PLAN
        # =====================================================

        print("\n🧠 STEP 1 — PLANNING")

        plan = self.plan(symbol)

        print(
            json.dumps(
                plan,
                indent=2,
                default=str,
            )
        )

        # =====================================================
        # 2. TOOL EXECUTION
        # =====================================================

        print("\n🔧 STEP 2 — DYNAMIC TOOL USE")

        observations = self.execute_tools(
            symbol,
            plan,
        )

        print("\n📊 RAW OBSERVATIONS")

        print(
            json.dumps(
                observations,
                indent=2,
                default=str,
            )
        )
                
        # =====================================================
        # 3. DATA VALIDATION
        # =====================================================

        print("\n🛡️ STEP 3 — DATA VALIDATION")

        validation_issues = self.validate_observations(
            symbol,
            observations,
        )

        if validation_issues:
            print(
                json.dumps(
                    validation_issues,
                    indent=2,
                    default=str,
                )
            )
        else:
            print("No automated validation issues found.")

        # =====================================================
        # 4. SELF REFLECTION
        # =====================================================

        print("\n🔍 STEP 4 — SELF-REFLECTION")

        reflection = self.reflect(
            symbol,
            plan,
            observations,
            validation_issues,
        )

        print(
            json.dumps(
                reflection,
                indent=2,
                default=str,
            )
        )

       # =====================================================
       # 5. REPORT
       # =====================================================

        print("\n📝 STEP 5 — REPORT GENERATION")

        report = self.report(
            symbol,
            plan,
            observations,
            reflection,
            validation_issues,
       )

        print("\n--- GENERATED REPORT ---")

        if report:
             print(report)
        else:
            print("❌ REPORT IS EMPTY")

            print("--- END GENERATED REPORT ---")


        # =====================================================
        # 6. REPORT VALIDATION
        # =====================================================

        print("\n🔎 STEP 6 — REPORT VALIDATION")

        report_issues = self.validate_report(
            symbol,
            report,
            observations,
        )

        if report_issues:
            for issue in report_issues:
                print(f"⚠️ {issue}")
        else:
            print("Report passed validation.")

        # =====================================================
        # 7. MEMORY
        # =====================================================

        print("\n💾 STEP 7 — LEARNING / MEMORY")

        self.remember(
            symbol,
            report,
            reflection,
        )

        print("Memory updated.")

        # =====================================================
        # STRUCTURED RESULT
        # =====================================================

        research_result = {
            "symbol": symbol,
            "plan": plan,
            "observations": observations,
            "validation": validation_issues,
            "reflection": reflection,
            "report": report,
            "report_validation": report_issues,
        }

        # =====================================================
        # CRISP CONSOLE REPORT
        # =====================================================

        self.print_final_report(
            research_result
        )

        return research_result


# =============================================================
# CREATE AGENT
# =============================================================

agent = MarketResearchAgent()

print("Agent created successfully")


# =============================================================
# MAIN PROGRAM
# =============================================================

def main():
    ticker = input(
        "Enter stock ticker (e.g., AAPL, MSFT, NVDA): "
    ).strip().upper()

    if not ticker:
        print("❌ No ticker entered.")
        return

    try:
        # -----------------------------------------------------
        # Verify ticker with yfinance before starting agent
        # -----------------------------------------------------

        print(f"\n🔎 Checking ticker: {ticker}")

        test = yf.Ticker(ticker)
        history = test.history(period="5d")

        if history.empty:
            print(
                f"❌ Could not find market data for '{ticker}'."
            )
            return

        print(
            f"✅ Found {ticker}. Starting research...\n"
        )

        # -----------------------------------------------------
        # Run agent
        # -----------------------------------------------------

        research_result = agent.run(ticker)

        # -----------------------------------------------------
        # Display LLM-generated Markdown report
        # -----------------------------------------------------

        print("\n" + "=" * 70)
        print("LLM RESEARCH REPORT")
        print("=" * 70)

        print(
            research_result.get(
                "report",
                "No report was generated.",
            )
        )

    except Exception as e:
        print(
            f"❌ Error checking or researching "
            f"ticker '{ticker}': {type(e).__name__}: {e}"
        )


# =============================================================
# ENTRY POINT
# =============================================================

if __name__ == "__main__":
    main()


 Team Members: Brandon, Christina and Isaac

Build an autonomous Investment Research Agent that:  
 1. Plans its research steps for a given stock symbol.
 2. Uses tools dynamically (APIs, datasets, retrieval). 
 3. Self-reflects to assess the quality of its output.  
 4. Learns across runs (e.g., keeps brief memories or notes to improve future analyses).  

Agent created successfully


Enter stock ticker (e.g., AAPL, MSFT, NVDA):  MSFT



🔎 Checking ticker: MSFT
✅ Found MSFT. Starting research...

AGENTIC MARKET RESEARCH: MSFT

🧠 STEP 1 — PLANNING
{
  "objectives": [
    "Understand the current financial health of MSFT",
    "Analyze the company's recent cash flow"
  ],
  "tools": [
    "financials",
    "cash_flow"
  ],
  "questions": [
    "What is MSFT's current revenue and profit?",
    "What is MSFT's recent cash flow pattern?"
  ]
}

🔧 STEP 2 — DYNAMIC TOOL USE
🔧 Running tool: financials
🔧 Running tool: cash_flow

📊 RAW OBSERVATIONS
{
  "financials": {
    "Total Revenue": {
      "2026-06-30 00:00:00": 331839000000.0,
      "2025-06-30 00:00:00": 281724000000.0,
      "2024-06-30 00:00:00": 245122000000.0,
      "2023-06-30 00:00:00": 211915000000.0
    },
    "Operating Income": {
      "2026-06-30 00:00:00": 155237000000.0,
      "2025-06-30 00:00:00": 128528000000.0,
      "2024-06-30 00:00:00": 109433000000.0,
      "2023-06-30 00:00:00": 88523000000.0
    },
    "Net Income": {
      "2026-06-30 00:00:00": 

In [19]:
#
print("\033[1m Team Member: Brandon\033[0m")
print()
print("\033[1mTask: yfinance pandas \033[0m")
print()
#

pip install yfinance pandas


In [22]:
#
print("\033[1m Team Members: Brandon, Christina and Isaac\033[0m")
print()
print("\033[1m 1. Prompt Chaining: Ingest News → Preprocess → Classify → Extract → Summarize  \033[0m")
print("\033[1m 2. Routing: Direct content to the right specialist (e.g., earnings, news, or market analyzers). \033[0m")
print("\033[1m 3. Evaluator–Optimizer: Generate analysis → evaluate quality → refine using feedback. \033[0m")
print()
#
"""
Agentic Stock News Analysis Pipeline

Demonstrates:

1. Prompt Chaining
   News -> Preprocess -> Classify -> Extract -> Summarize

2. Routing
   Article -> Earnings Specialist
           -> News Specialist
           -> Market Specialist

3. Evaluator-Optimizer
   Generate Analysis -> Evaluate -> Refine

Input:
    Stock ticker, e.g. AAPL, MSFT, NVDA

Note:
    This is an educational analysis pipeline.
    It does not provide investment recommendations.
"""

import re
import json
from datetime import datetime

import yfinance as yf


# ============================================================
# CONFIGURATION
# ============================================================

MAX_NEWS_ARTICLES = 8


# ============================================================
# STEP 0 — INGEST NEWS
# ============================================================

def ingest_news(ticker_symbol, max_articles=MAX_NEWS_ARTICLES):
    """
    Retrieve recent news for a stock ticker.

    yfinance provides Ticker.get_news(count=...)
    """

    ticker = yf.Ticker(ticker_symbol)

    try:
        raw_news = ticker.get_news(count=max_articles)
    except Exception as e:
        print(f"ERROR: Could not retrieve news: {e}")
        return []

    articles = []

    for item in raw_news:

        # yfinance/Yahoo Finance responses can vary slightly.
        content = item.get("content", item)

        title = (
            content.get("title")
            or item.get("title")
            or "Unknown title"
        )

        summary = (
            content.get("summary")
            or item.get("summary")
            or ""
        )

        description = (
            content.get("description")
            or item.get("description")
            or ""
        )

        # Try several possible locations for the URL.
        url = (
            content.get("canonicalUrl", {}).get("url")
            if isinstance(content.get("canonicalUrl"), dict)
            else None
        )

        if not url:
            url = (
                content.get("clickThroughUrl", {}).get("url")
                if isinstance(content.get("clickThroughUrl"), dict)
                else None
            )

        if not url:
            url = item.get("link")

        publisher = (
            content.get("provider", {}).get("displayName")
            if isinstance(content.get("provider"), dict)
            else None
        )

        articles.append({
            "title": title,
            "summary": summary,
            "description": description,
            "publisher": publisher or "Unknown",
            "url": url or "N/A"
        })

    return articles


# ============================================================
# PROMPT CHAIN — STEP 1
# PREPROCESS
# ============================================================

def preprocess_article(article):
    """
    Clean and normalize article text.
    """

    text = " ".join([
        article.get("title", ""),
        article.get("summary", ""),
        article.get("description", "")
    ])

    # Remove excessive whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    # Basic normalization.
    normalized = text.lower()

    return {
        **article,
        "raw_text": text,
        "normalized_text": normalized
    }


# ============================================================
# PROMPT CHAIN — STEP 2
# CLASSIFY
# ============================================================

def classify_article(article):
    """
    Classify an article into one of three specialist categories.

    In a production LLM implementation, this function could
    be replaced with an LLM classifier.
    """

    text = article["normalized_text"]

    earnings_keywords = [
        "earnings",
        "revenue",
        "eps",
        "profit",
        "quarter",
        "guidance",
        "forecast",
        "financial results",
        "net income"
    ]

    market_keywords = [
        "stock",
        "shares",
        "market",
        "price",
        "downgrade",
        "upgrade",
        "analyst",
        "valuation",
        "trading"
    ]

    earnings_score = sum(
        1 for keyword in earnings_keywords
        if keyword in text
    )

    market_score = sum(
        1 for keyword in market_keywords
        if keyword in text
    )

    if earnings_score > market_score and earnings_score > 0:
        category = "earnings"

    elif market_score > 0:
        category = "market"

    else:
        category = "news"

    return {
        **article,
        "category": category,
        "classification_scores": {
            "earnings": earnings_score,
            "market": market_score
        }
    }


# ============================================================
# PROMPT CHAIN — STEP 3
# EXTRACT
# ============================================================

def extract_information(article):
    """
    Extract structured information from the article.

    This is deliberately simple so that the pipeline can run
    without requiring an external LLM API.
    """

    text = article["raw_text"]

    extracted = {
        "companies": [],
        "financial_numbers": [],
        "events": [],
        "keywords": []
    }

    # Detect common financial numbers.
    financial_patterns = [
        r"\$\d+(?:\.\d+)?(?:\s?(?:billion|million|trillion|B|M|T))?",
        r"\d+(?:\.\d+)?%",
        r"\d+(?:\.\d+)?\s?(?:million|billion|trillion)"
    ]

    for pattern in financial_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        for match in matches:
            if match not in extracted["financial_numbers"]:
                extracted["financial_numbers"].append(match)

    # Basic event detection.
    event_keywords = [
        "earnings",
        "acquisition",
        "merger",
        "layoffs",
        "product launch",
        "guidance",
        "partnership",
        "lawsuit",
        "dividend",
        "buyback"
    ]

    lower_text = text.lower()

    for event in event_keywords:
        if event in lower_text:
            extracted["events"].append(event)

    # Important keywords.
    keyword_candidates = [
        "revenue",
        "earnings",
        "eps",
        "profit",
        "growth",
        "guidance",
        "analyst",
        "market",
        "shares",
        "AI",
        "cloud",
        "acquisition"
    ]

    for keyword in keyword_candidates:
        if keyword.lower() in lower_text:
            extracted["keywords"].append(keyword)

    return {
        **article,
        "extracted": extracted
    }


# ============================================================
# PROMPT CHAIN — STEP 4
# SUMMARIZE
# ============================================================

def summarize_article(article):
    """
    Create a compact summary.

    A production system could replace this with an LLM.
    """

    title = article["title"]
    category = article["category"]
    extracted = article["extracted"]

    events = ", ".join(extracted["events"]) or "none detected"
    numbers = ", ".join(extracted["financial_numbers"]) or "none detected"

    summary = (
        f"The article concerns {category}-related information. "
        f"Detected events: {events}. "
        f"Financial figures mentioned: {numbers}. "
        f"Headline: {title}."
    )

    return {
        **article,
        "summary_output": summary
    }


# ============================================================
# ROUTING
# ============================================================

def route_to_specialist(article):
    """
    Route the article to the appropriate specialist.
    """

    specialists = {
        "earnings": earnings_specialist,
        "market": market_specialist,
        "news": news_specialist
    }

    specialist = specialists.get(
        article["category"],
        news_specialist
    )

    analysis = specialist(article)

    return {
        **article,
        "specialist": specialist.__name__,
        "specialist_analysis": analysis
    }


# ============================================================
# SPECIALIST 1 — EARNINGS
# ============================================================

def earnings_specialist(article):
    """
    Analyze earnings-related information.
    """

    extracted = article["extracted"]

    return {
        "focus": "earnings",
        "observations": [
            f"Financial figures detected: "
            f"{', '.join(extracted['financial_numbers']) or 'none'}",
            f"Events detected: "
            f"{', '.join(extracted['events']) or 'none'}"
        ],
        "interpretation": (
            "This article should be reviewed primarily for "
            "changes in revenue, EPS, profitability, and guidance."
        )
    }


# ============================================================
# SPECIALIST 2 — MARKET
# ============================================================

def market_specialist(article):
    """
    Analyze market-related information.
    """

    extracted = article["extracted"]

    return {
        "focus": "market",
        "observations": [
            f"Market keywords: "
            f"{', '.join(extracted['keywords']) or 'none'}",
            f"Financial figures: "
            f"{', '.join(extracted['financial_numbers']) or 'none'}"
        ],
        "interpretation": (
            "This article should be reviewed for potential "
            "market, valuation, analyst, or share-price implications."
        )
    }


# ============================================================
# SPECIALIST 3 — GENERAL NEWS
# ============================================================

def news_specialist(article):
    """
    Analyze general company news.
    """

    extracted = article["extracted"]

    return {
        "focus": "general_news",
        "observations": [
            f"Events: "
            f"{', '.join(extracted['events']) or 'none'}",
            f"Keywords: "
            f"{', '.join(extracted['keywords']) or 'none'}"
        ],
        "interpretation": (
            "This article should be reviewed for significant "
            "company developments and their potential relevance."
        )
    }


# ============================================================
# EVALUATOR
# ============================================================

def evaluate_analysis(article):
    """
    Evaluate the generated analysis.

    Scores:
        completeness
        relevance
        evidence
        clarity

    Score range: 0-100
    """

    analysis = article["specialist_analysis"]

    score = 0
    feedback = []

    # Completeness
    if analysis.get("observations"):
        score += 25
    else:
        feedback.append("Add concrete observations.")

    # Relevance
    if analysis.get("focus"):
        score += 25
    else:
        feedback.append("Specify analysis focus.")

    # Evidence
    observations = analysis.get("observations", [])

    if len(observations) >= 2:
        score += 25
    else:
        feedback.append("Include more evidence from extracted data.")

    # Clarity
    if analysis.get("interpretation"):
        score += 25
    else:
        feedback.append("Add a clear interpretation.")

    # Quality threshold
    passed = score >= 75

    return {
        "score": score,
        "passed": passed,
        "feedback": feedback
    }


# ============================================================
# OPTIMIZER
# ============================================================

def optimize_analysis(article, evaluation):
    """
    Refine the generated analysis using evaluator feedback.
    """

    analysis = article["specialist_analysis"].copy()

    if evaluation["score"] < 75:

        observations = analysis.get("observations", [])

        observations.append(
            "Optimizer feedback: "
            + " ".join(evaluation["feedback"])
        )

        analysis["observations"] = observations

        analysis["interpretation"] = (
            analysis.get("interpretation", "")
            + " The refined analysis explicitly incorporates "
              "the extracted evidence and evaluator feedback."
        )

    else:

        analysis["interpretation"] = (
            analysis.get("interpretation", "")
            + " Analysis passed the initial quality evaluation."
        )

    return {
        **article,
        "optimized_analysis": analysis
    }


# ============================================================
# DISPLAY HELPERS
# ============================================================

def print_separator():
    print("\n" + "=" * 80)


def print_article_result(article, number):
    print_separator()

    print(f"ARTICLE {number}")
    print(f"Title       : {article['title']}")
    print(f"Publisher   : {article['publisher']}")
    print(f"URL         : {article['url']}")

    print("\n[1] PREPROCESS")
    print(f"Clean text  : {article['raw_text'][:300]}...")

    print("\n[2] CLASSIFY")
    print(f"Category    : {article['category']}")
    print(
        f"Scores      : {article['classification_scores']}"
    )

    print("\n[3] EXTRACT")
    print(
        json.dumps(
            article["extracted"],
            indent=2
        )
    )

    print("\n[4] SUMMARIZE")
    print(article["summary_output"])

    print("\n[5] ROUTING")
    print(f"Specialist  : {article['specialist']}")

    print("\n[6] SPECIALIST ANALYSIS")

    specialist = article["specialist_analysis"]

    print(f"Focus       : {specialist['focus']}")

    for observation in specialist["observations"]:
        print(f"  - {observation}")

    print(f"Interpretation: {specialist['interpretation']}")

    print("\n[7] EVALUATOR")

    evaluation = article["evaluation"]

    print(f"Quality score: {evaluation['score']}/100")
    print(f"Passed       : {evaluation['passed']}")

    if evaluation["feedback"]:
        print("Feedback:")
        for feedback in evaluation["feedback"]:
            print(f"  - {feedback}")

    print("\n[8] OPTIMIZER")

    optimized = article["optimized_analysis"]

    print("Refined analysis:")

    for observation in optimized["observations"]:
        print(f"  - {observation}")

    print(f"Interpretation: {optimized['interpretation']}")


# ============================================================
# COMPLETE PIPELINE
# ============================================================

def analyze_stock(ticker_symbol):
    """
    Run the entire agentic pipeline.
    """

    ticker_symbol = ticker_symbol.upper().strip()

    print_separator()
    print("AGENTIC STOCK NEWS ANALYSIS")
    print(f"Ticker: {ticker_symbol}")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # --------------------------------------------------------
    # INGEST
    # --------------------------------------------------------

    print_separator()
    print("STEP 0 — INGEST NEWS")

    articles = ingest_news(ticker_symbol)

    if not articles:
        print("No news articles were retrieved.")
        return

    print(f"Retrieved {len(articles)} articles.")

    processed_articles = []

    # --------------------------------------------------------
    # PROMPT CHAIN
    # --------------------------------------------------------

    for article in articles:

        # 1. Preprocess
        article = preprocess_article(article)

        # 2. Classify
        article = classify_article(article)

        # 3. Extract
        article = extract_information(article)

        # 4. Summarize
        article = summarize_article(article)

        # ----------------------------------------------------
        # ROUTING
        # ----------------------------------------------------

        article = route_to_specialist(article)

        # ----------------------------------------------------
        # EVALUATOR
        # ----------------------------------------------------

        evaluation = evaluate_analysis(article)

        article = {
            **article,
            "evaluation": evaluation
        }

        # ----------------------------------------------------
        # OPTIMIZER
        # ----------------------------------------------------

        article = optimize_analysis(
            article,
            evaluation
        )

        processed_articles.append(article)

    # --------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------

    for i, article in enumerate(
        processed_articles,
        start=1
    ):
        print_article_result(article, i)

    # --------------------------------------------------------
    # FINAL PIPELINE SUMMARY
    # --------------------------------------------------------

    print_separator()
    print("PIPELINE COMPLETE")

    print(f"Ticker: {ticker_symbol}")
    print(f"Articles processed: {len(processed_articles)}")

    print("\nRouting summary:")

    routing_counts = {}

    for article in processed_articles:

        category = article["category"]

        routing_counts[category] = (
            routing_counts.get(category, 0) + 1
        )

    for category, count in routing_counts.items():
        print(f"  {category}: {count}")

    average_score = (
        sum(
            article["evaluation"]["score"]
            for article in processed_articles
        )
        / len(processed_articles)
    )

    print(
        f"\nAverage evaluator score: "
        f"{average_score:.1f}/100"
    )

    print_separator()


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    ticker = input(
        "Enter stock ticker (e.g., AAPL, MSFT, NVDA): "
    )

    if not ticker.strip():
        print("Ticker cannot be empty.")
    else:
        analyze_stock(ticker)


 Team Members: Brandon, Christina and Isaac

 1. Prompt Chaining: Ingest News → Preprocess → Classify → Extract → Summarize  
 2. Routing: Direct content to the right specialist (e.g., earnings, news, or market analyzers). 
 3. Evaluator–Optimizer: Generate analysis → evaluate quality → refine using feedback. 



Enter stock ticker (e.g., AAPL, MSFT, NVDA):  NVDA



AGENTIC STOCK NEWS ANALYSIS
Ticker: NVDA
Started: 2026-09-22 19:29:10

STEP 0 — INGEST NEWS
Retrieved 8 articles.

ARTICLE 1
Title       : Should investors play AI bottlenecks over social media stocks?
Publisher   : Yahoo Finance Video
URL         : https://finance.yahoo.com/video/investors-play-ai-bottlenecks-over-173922161.html

[1] PREPROCESS
Clean text  : Should investors play AI bottlenecks over social media stocks? Should investors bet more on the AI infrastructure bottlenecks than on other tech trends, such as social media? Today's Market Hang panel consists of Host Turney Duff, Yahoo Finance Executive Editor Brian Sozzi, Defiance ETFs CIO Sylvia ...

[2] CLASSIFY
Category    : market
Scores      : {'earnings': 0, 'market': 2}

[3] EXTRACT
{
  "companies": [],
  "financial_numbers": [],
  "events": [],
  "keywords": [
    "market",
    "AI"
  ]
}

[4] SUMMARIZE
The article concerns market-related information. Detected events: none detected. Financial figures mentioned: none det